<a href="https://colab.research.google.com/github/Not-kh-lily-23/pulsar-conformal-triage/blob/main/medlat_replication.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import lightgbm as lgb
from google.colab import drive
drive.mount('/content/drive')
medlat_path='/content/drive/MyDrive/pulsar_project/data/processed/medlat_full.csv'
df=pd.read_csv(medlat_path).dropna()
x=df.drop(columns=['target'])
y=df['target']
print(f"total number of rows are {len(df)}")
print(f"positive class ration: {(y.sum()/len(y))*100:.2f}")
x_temp,x_test,y_temp,y_test=train_test_split(x,y,test_size=0.20,random_state=42,stratify=y)
x_train,x_calib,y_train,y_calib=train_test_split(x_temp,y_temp,test_size=0.25,random_state=42,stratify=y_temp)
print(f"splits to train: {len(y_train)} | caliberation: {len(y_calib)} | test: {len(y_test)}")
clf=lgb.LGBMClassifier(random_state=42,n_estimators=100,verbose=-1)
clf.fit(x_train,y_train)
print("done")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
total number of rows are 91192
positive class ration: 1.31
splits to train: 54714 | caliberation: 18239 | test: 18239
done


In [7]:
import numpy as np
calib_prob=clf.predict_proba(x_calib)
test_prob=clf.predict_proba(x_test)
calib_scores=1-calib_prob[np.arange(len(y_calib)),y_calib]
test_scores_true=1-test_prob[np.arange(len(y_test)), y_test]
alpha=0.05
target_cov=1-alpha
n_calib=len(calib_scores)
q_marg=np.quantile(calib_scores,min(1.0,np.ceil((n_calib+1)*target_cov)/n_calib))
marg_cov_0=np.mean(test_scores_true[y_test==0]<=q_marg)
marg_cov_1=np.mean(test_scores_true[y_test==1]<=q_marg)
scores_0=calib_scores[y_calib==0]
scores_1=calib_scores[y_calib==1]
n_0=len(scores_0)
n_1=len(scores_1)
q_0=np.quantile(scores_0,min(1.0,np.ceil((n_0+1)*target_cov)/n_0))
q_1=np.quantile(scores_1,min(1.0,np.ceil((n_1+1)*target_cov)/n_1))
mondrian_cov_0=np.mean(test_scores_true[y_test==0]<=q_0)
mondrian_cov_1=np.mean(test_scores_true[y_test==1]<=q_1)
print(f"target discovery floor: {target_cov:.4f}")
print("marginal cp (standard model):")
print(f"class 0 (noise) coverage: {marg_cov_0:.4f}")
print(f"class 1 (pulsar) coverage: {marg_cov_1:.4f}")
print("mondrian cp (class conditional):")
print(f"class 0 (noise) coverage: {mondrian_cov_0:.4f}")
print(f"class 1 (pulsar) xoverage: {mondrian_cov_1:.4f}")

target discovery floor: 0.9500
marginal cp (standard model):
class 0 (noise) coverage: 0.9624
class 1 (pulsar) coverage: 0.0251
mondrian cp (class conditional):
class 0 (noise) coverage: 0.9506
class 1 (pulsar) xoverage: 0.9540


In [5]:
import numpy as np
from sklearn.model_selection import train_test_split
import lightgbm as lgb
import scipy.stats as st
n_seeds=20
alpha=0.05
target_cov=1-alpha
mond_cov_1_list=[]
marg_cov_1_list=[]
for seed in range(n_seeds):
    x_temp,x_test,y_temp,y_test=train_test_split(x,y,test_size=0.20,random_state=seed,stratify=y)
    x_train,x_calib,y_train,y_calib=train_test_split(x_temp,y_temp,test_size=0.25,random_state=seed,stratify=y_temp)
    clf=lgb.LGBMClassifier(random_state=seed,n_estimators=100,verbose=-1)
    clf.fit(x_train,y_train)
    calib_prob=clf.predict_proba(x_calib)
    test_prob=clf.predict_proba(x_test)
    calib_scores=1-calib_prob[np.arange(len(y_calib)),y_calib]
    test_scores_true=1-test_prob[np.arange(len(y_test)),y_test]
    n_calib=len(calib_scores)
    q_marg=np.quantile(calib_scores,min(1.0,np.ceil((n_calib+1)*target_cov)/n_calib))
    marg_cov_1_list.append(np.mean(test_scores_true[y_test==1]<=q_marg))
    scores_1=calib_scores[y_calib==1]
    n_1=len(scores_1)
    if n_1>0:
        q_1=np.quantile(scores_1,min(1.0,np.ceil((n_1+1)*target_cov)/n_1))
        mond_cov_1_list.append(np.mean(test_scores_true[y_test==1]<=q_1))
    if (seed+1)%5==0:
        print(f"completed seed{seed+1}/{n_seeds}")
def calc_ci(data):
    mean=np.mean(data)
    ci=st.t.interval(0.95,len(data)-1,loc=mean,scale=st.sem(data))
    return mean,ci[0],ci[1]
marg_mean,marg_lower,marg_upper=calc_ci(marg_cov_1_list)
mond_mean,mond_lower,mond_upper=calc_ci(mond_cov_1_list)
print(f"target coverage: {target_cov:.4f}")
print("marginal cp (class 1 - pulsar):")
print(f"mean Coverage: {marg_mean:.4f} [95% CI: {marg_lower:.4f} - {marg_upper:.4f}]")
print("mondrian cp (class 1 - pulsar):")
print(f"mean Coverage: {mond_mean:.4f} [95% CI: {mond_lower:.4f} - {mond_upper:.4f}]")

completed seed5/20
completed seed10/20
completed seed15/20
completed seed20/20
target coverage: 0.9500
marginal cp (class 1 - pulsar):
mean Coverage: 0.0188 [95% CI: 0.0080 - 0.0297]
mondrian cp (class 1 - pulsar):
mean Coverage: 0.9473 [95% CI: 0.9377 - 0.9569]
